<a href="https://colab.research.google.com/github/BilalKhaliqWillis/BILAL-Assignment2/blob/main/BILAL_Assignment_2_Containerization_and_Edge_Computing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2: Containerization and Edge Computing
# Flask + Image Processing + TensorFlow Lite + Docker


In [1]:

!pip install flask numpy pillow tensorflow


In [2]:

import base64
import numpy as np
from PIL import Image
import io
import tensorflow as tf


# Step 1: Create TFLite Model

In [3]:

model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28)),
    tf.keras.layers.Dense(10, activation='softmax')
])

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_model)

print("Model saved")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Saved artifact at '/tmp/tmpmqowa2mg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  135115200833168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135115200834128: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model saved


# Step 2: Flask App

In [4]:

%%writefile app.py
from flask import Flask, request, jsonify
import base64
import numpy as np
from PIL import Image
import io
import tensorflow as tf

app = Flask(__name__)

interpreter = tf.lite.Interpreter(model_path="model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

@app.route('/')
def home():
    return "Edge ML App Running!"

@app.route('/predict_image', methods=['POST'])
def predict_image():
    data = request.json['image']
    image_bytes = base64.b64decode(data)
    image = Image.open(io.BytesIO(image_bytes)).convert('L')
    image = image.resize((28,28))
    image = np.array(image)/255.0
    image = image.reshape(1,28,28)

    interpreter.set_tensor(input_details[0]['index'], image.astype(np.float32))
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]['index'])
    prediction = int(np.argmax(output))

    return jsonify({"prediction": prediction})

if __name__ == "__main__":
    app.run()


Writing app.py


# Step 3: Dockerfile

In [5]:

%%writefile Dockerfile
FROM python:3.9
WORKDIR /app
COPY . .
RUN pip install flask numpy pillow tensorflow
CMD ["python", "app.py"]


Writing Dockerfile


In [7]:
!python app.py

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


VMs vs Containers
•	VMs use a full operating system → heavy and slow
•	Containers share the OS → lightweight and fast
  - Containers use fewer resources than VMs.

2. Role of Docker
Docker packages an app with all its dependencies into a container.
Advantages:
•	Runs the same everywhere
•	Easy to deploy
•	No environment issues

3. Edge vs Cloud Computing
•	Edge computing: processes data near the device (fast, real-time)
•	Cloud computing: processes data on remote servers (powerful but slower)

4. Steps to create Docker container
1.	Create Python ML app
2.	Write Dockerfile
3.	Install dependencies
4.	Build image
5.	Run container
Main Dockerfile parts:
FROM, COPY, RUN, CMD

5. Container Registry (Docker Hub)
Stores and shares Docker images.
•	Push: upload image
•	Pull: download image
 - Helps in easy deployment and sharing.

